In [3]:
# Install necessary libraries if not already installed
# pip install transformers datasets

# Import necessary libraries
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Add a padding token to the tokenizer
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Resize the model's token embeddings to account for the new token
model.resize_token_embeddings(len(tokenizer))

# Load the CSV file into a Hugging Face dataset
dataset = load_dataset('csv', data_files='sample_gpt2_training_data.csv')['train']

# Tokenize the dataset
def tokenize_function(examples):
    # Tokenize the text and add the labels for language modeling
    encodings = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    encodings['labels'] = encodings['input_ids']  # Set labels for language modeling
    return encodings

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2-finetuned")


Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Step,Training Loss
10,5.107900


In [10]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the fine-tuned model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('./gpt2-finetuned')
model = GPT2LMHeadModel.from_pretrained('./gpt2-finetuned')
import torch

def generate_text(prompt, max_length=50, num_return_sequences=1):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # Generate text
    outputs = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        no_repeat_ngram_size=2,  # To prevent repetition
        top_k=50,                # Top-k sampling
        top_p=0.95                # Top-p sampling (nucleus sampling)
    )
    
    # Decode the generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

# Example usage
prompt = "fahad is"
generated_texts = generate_text(prompt)
print(generated_texts)



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['fahad is the of " ( ). =\n._,,; ; } :: () { _ (); __ ().']
